# Hand Gesture Recognition — Exploration Notebook
This notebook guides you through the full pipeline:
1. Dataset inspection
2. Feature visualization
3. Model training & evaluation
4. Testing on static images

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import json
from collections import Counter

from gesture_recognizer import GestureRecognizer, GESTURE_LABELS

print('✅ Imports successful')

## 1. Dataset Inspection

In [ ]:
# Load dataset
DATASET_PATH = '../data/raw/dataset.json'

if not os.path.exists(DATASET_PATH):
    print('⚠ No dataset found. Run collect_data.py first.')
    print('  Example: python src/collect_data.py --gesture thumbs_up --samples 200')
else:
    with open(DATASET_PATH) as f:
        data = json.load(f)

    X = np.array(data['X'])
    y = np.array(data['y'])
    inv_map = {v: k for k, v in data['gesture_map'].items()}

    print(f'Dataset shape: {X.shape}')
    print(f'Features per sample: {X.shape[1]}')
    print(f'\nClass distribution:')
    for label, count in sorted(Counter(y).items()):
        print(f'  {inv_map[label]}: {count} samples')

In [ ]:
# Visualize class distribution
counts = Counter(y)
labels = [inv_map[k] for k in sorted(counts.keys())]
values = [counts[k] for k in sorted(counts.keys())]
colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
ax.set_title('Gesture Class Distribution', fontsize=16, fontweight='bold')
ax.set_ylabel('Sample Count')
ax.set_xlabel('Gesture')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', fontweight='bold')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Visualization (PCA / t-SNE)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
unique_labels = sorted(set(y))
cmap = plt.cm.get_cmap('tab10', len(unique_labels))

for i, label in enumerate(unique_labels):
    mask = y == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[cmap(i)], label=inv_map[label],
               alpha=0.6, s=30, edgecolors='none')

ax.set_title('PCA Visualization of Gesture Features', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()
print(f'Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 3. Model Training

In [ ]:
from train_model import train

# Train Random Forest
model_rf, acc_rf = train(
    model_type='random_forest',
    dataset_path=DATASET_PATH,
    output_path='../models/gesture_rf.pkl'
)
print(f'\nRandom Forest Accuracy: {acc_rf*100:.2f}%')

In [ ]:
# Train MLP
model_mlp, acc_mlp = train(
    model_type='mlp',
    dataset_path=DATASET_PATH,
    output_path='../models/gesture_mlp.pkl'
)
print(f'\nMLP Accuracy: {acc_mlp*100:.2f}%')

## 4. Test on a Static Image

In [ ]:
# Run recognizer on an image file
IMAGE_PATH = 'your_hand_image.jpg'  # ← Change this

recognizer = GestureRecognizer(model_path='../models/gesture_rf.pkl')
annotated, detections = recognizer.process_image(IMAGE_PATH)

if annotated is not None:
    # Display with matplotlib
    plt.figure(figsize=(10, 7))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Gesture Recognition Result')
    plt.show()

    for pred, label, bbox in detections:
        print(f'  Detected: {label}')